## 导入模块并初始化


In [1]:
from pathlib import Path
import pandas as pd
from market_data import create_manager, today_str, load_tushare_token

In [2]:
# Tushare Token
TUSHARE_TOKEN = load_tushare_token()

# 数据库路径：直接放在项目目录下
DB_PATH = "data/db/market_data.db"

manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date="20150101",
    default_exchange="SSE",
)

print("数据库位置：", Path(DB_PATH).resolve())

数据库位置： D:\codeWork\RiskParity\data\db\market_data.db


## 2. 初始化基础数据
项目开始时先跑一次，用来获取：
场内基金基础信息
交易日历

In [ ]:
manager.initialize_basic_data(
    fund_market="E",      # 场内基金
    fund_status="L",      # 上市状态
    cal_start_date="20150101",
    cal_end_date=today_str(),
    exchange="SSE",
)

## 3. 查看当前本地资产列表

In [5]:
instruments = manager.store.get_instruments(listed_only=True)
print(instruments.shape)
instruments.head()

(1879, 27)


,ts_code,name,management,custodian,fund_type,found_date,due_date,list_date,issue_date,delist_date,...,benchmark,status,invest_type,type,trustee,purc_startdate,redm_startdate,market,created_at,updated_at
0,159001.SZ,货币ETF易方达,易方达基金,交通银行,货币市场型,20130329,None,20141020,20130325,None,...,活期存款基准利率*(1-利息税税率),L,货币型,契约型开放式,None,20130418,20130418,E,2026-03-24 20:28:52,2026-03-24 20:28:52
1,159003.SZ,招商快线ETF,招商基金,平安银行,货币市场型,20130517,None,20141020,20130513,None,...,人民币活期存款基准利率(税后),L,货币型,契约型开放式,None,20130607,20130607,E,2026-03-24 20:28:52,2026-03-24 20:28:52
2,159005.SZ,汇添富快钱ETF,汇添富基金,中国工商银行,货币市场型,20141223,None,20150113,20141215,None,...,活期存款利率(税后),L,货币型,契约型开放式,None,20150113,20150113,E,2026-03-24 20:28:52,2026-03-24 20:28:52
3,159100.SZ,巴西ETF,华夏基金,中国工商银行,股票型,20251105,None,20251113,20251031,None,...,巴西伊博维斯帕指数收益率(经估值汇率调整后),L,被动指数型,契约型开放式,None,20251113,20251113,E,2026-03-24 20:28:52,2026-03-24 20:28:52
4,159101.SZ,港股通科技ETF基金,华夏基金,中信银行,股票型,20250826,None,20250903,20250818,None,...,经估值汇率调整后的国证港股通科技指数收益率,L,被动指数型,契约型开放式,None,20250903,20250903,E,2026-03-24 20:28:52,2026-03-24 20:28:52


## 4. 查看本地已收录的 ts_code 列表

In [6]:
ts_codes = manager.store.get_ts_codes(listed_only=True)
len(ts_codes), ts_codes[:10]

(1879,
 ['159001.SZ',
  '159003.SZ',
  '159005.SZ',
  '159100.SZ',
  '159101.SZ',
  '159102.SZ',
  '159103.SZ',
  '159105.SZ',
  '159106.SZ',
  '159107.SZ'])

5. 更新单个基金/ETF 日线数据

例如“515100.SH”

In [3]:
df_515100 = manager.update_one_daily_price(
    ts_code="515100.SH",
    start_date="20180101",
    end_date=today_str(),
)
df_515100.tail()

,ts_code,trade_date,pre_close,open,high,low,close,change,pct_chg,vol,amount,source,created_at,updated_at
1383,515100.SH,20260319,1.482,1.480,1.490,1.475,1.479,-0.003,-0.2024,2782077.96,412239.982,tushare,2026-03-26 03:12:48,2026-03-26 03:12:48
1384,515100.SH,20260320,1.479,1.477,1.488,1.471,1.473,-0.006,-0.4057,826949.74,122382.826,tushare,2026-03-26 03:12:48,2026-03-26 03:12:48
1385,515100.SH,20260323,1.473,1.467,1.469,1.417,1.424,-0.049,-3.3265,2059811.82,296113.965,tushare,2026-03-26 03:12:48,2026-03-26 03:12:48
1386,515100.SH,20260324,1.424,1.430,1.448,1.425,1.448,0.024,1.6854,1160140.24,166530.521,tushare,2026-03-26 03:12:48,2026-03-26 03:12:48
1387,515100.SH,20260325,1.448,1.449,1.463,1.439,1.461,0.013,0.8978,1112365.58,161939.432,tushare,2026-03-26 03:12:48,2026-03-26 03:12:48


## 6. 批量更新指定资产池

In [ ]:
watchlist = [
    "510300.SH",  # 沪深300ETF
    "511010.SH",  # 国债ETF
    "518880.SH",  # 黄金ETF
]

summary = manager.update_daily_prices(
    ts_codes=watchlist,
    start_date="20150101",
    end_date=today_str(),
)

summary

## 7. 对全库做增量更新

In [ ]:
summary_all = manager.update_daily_prices()
summary_all

## 8. 只刷新最近一段时间的数据
这个功能适合日常维护。
比如你已经有历史全量数据了，之后每天只想回补最近 30 天或 60 天。

In [ ]:
recent_summary = manager.refresh_recent_daily_prices(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"],
    lookback_days=30,
    end_date=today_str(),
)
recent_summary

如果想对全体上市基金刷新最近 30 天：

In [ ]:
recent_summary_all = manager.refresh_recent_daily_prices(
    lookback_days=30,
    end_date=today_str(),
)
recent_summary_all

## 9. 检查本地价格数据覆盖范围

In [ ]:
coverage = manager.check_price_coverage(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"]
)
coverage

检查全库：

In [ ]:
coverage_all = manager.check_price_coverage()
coverage_all.head(20)

筛选出还没有数据的标的：

In [ ]:
coverage_all[coverage_all["has_data"] == 0].head(20)

## 10. 检查数据是否已经更新到最新交易日

In [ ]:
status = manager.check_latest_price_status(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"],
    exchange="SSE",
)
status

筛选出未更新到最新交易日的资产：

In [ ]:
status[status["is_up_to_date"] == 0]

## 11. 读取本地已存储的日线数据
 读取单个标的

In [ ]:
prices_510300 = manager.store.get_daily_prices(
    ts_codes=["510300.SH"],
    start_date="20230101",
    end_date=today_str(),
)
prices_510300.tail()

读取多个标的

In [ ]:
prices = manager.store.get_daily_prices(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"],
    start_date="20230101",
    end_date=today_str(),
)
prices.head()

## 12. 透视查看单个字段（比如 close）

In [ ]:
prices = manager.store.get_daily_prices(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"],
    start_date="20230101",
    end_date=today_str(),
)

close_df = prices.pivot(index="trade_date", columns="ts_code", values="close").sort_index()
close_df.tail()

成交额矩阵

In [ ]:
amount_df = prices.pivot(index="trade_date", columns="ts_code", values="amount").sort_index()
amount_df.tail()

## 13. 查看交易日历

In [ ]:
trade_cal = manager.store.get_trade_calendar(
    exchange="SSE",
    start_date="20240101",
    end_date=today_str(),
)
trade_cal.tail()

只看开市日

In [ ]:
open_dates = manager.store.get_open_trade_dates(
    exchange="SSE",
    start_date="20240101",
    end_date=today_str(),
)
open_dates[-10:]

## 14. 查看某个标的的数据起止日期

In [ ]:
manager.store.get_price_date_range("510300.SH")

查看某个标的最新交易日：

In [ ]:
manager.store.get_latest_trade_date("510300.SH")

## 15. 查看哪些标的还没有本地日线数据

In [ ]:
missing_codes = manager.store.get_missing_ts_codes(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"]
)
missing_codes

查看全库

In [ ]:
missing_all = manager.store.get_missing_ts_codes()
len(missing_all)

## 16. 查看哪些标的数据为空或滞后
比如要所有标的至少更新到某天：

In [ ]:
stale_codes = manager.store.get_empty_or_stale_ts_codes(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"],
    expected_latest_trade_date="20260320",
)
stale_codes

## 17.查看更新日志
查看最近的

In [ ]:
logs = manager.store.get_update_log(limit=20)
logs

只看日线更新日志：

In [ ]:
logs_daily = manager.store.get_update_log(
    table_name="daily_prices",
    limit=20,
)
logs_daily

只看某个标的

In [ ]:
logs_510300 = manager.store.get_update_log(
    table_name="daily_prices",
    ts_code="510300.SH",
    limit=20,
)
logs_510300

## 18. 最常用的日常维护流程示例

以后作为 notebook 的“每日更新入口”。

In [1]:
from pathlib import Path
import pandas as pd
from market_data import create_manager, today_str, load_tushare_token
# Tushare Token
TUSHARE_TOKEN = load_tushare_token()


# 数据库路径：直接放在项目目录下
DB_PATH = "data/db/market_data.db"

manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date="20070101",
    default_exchange="SSE",
)
watchlist = [
    "510300.SH",  # 沪深300ETF (核心权益：大盘蓝筹)
    "510500.SH",  # 中证500ETF (核心权益：中盘成长)
    "510170.SH",  # 50ETF (核心权益：大宗商品股票集合)
    "159915.SZ",  # 创业板ETF (核心权益：高新成长 - 纯被动)
    "511090.SH",  # 30年国债ETF (避险资产：超长债，对利率极度敏感)
    "511010.SH",  # 国债ETF (基础配置：中长期债)
    "518880.SH",  # 黄金ETF (抗通胀/避险：金属商品)
    "159981.SZ",  # 能源ETF (抗通胀：能源/电力)
    "159985.SZ",  # 豆粕ETF (抗通胀：农产品期货)
    "501018.SH",  # 南方原油LOF (抗通胀：国际原油价格)
    "515100.SH",  # 红利低波100ETF (抗通胀：红利股)
]

# 1. 刷新最近 30 天数据
recent_summary = manager.refresh_recent_daily_prices(
    ts_codes=watchlist,
    lookback_days=30,
    end_date=today_str(),
)

print("最近数据刷新结果：")
print(recent_summary)

# 2. 检查最新状态
status = manager.check_latest_price_status(
    ts_codes=watchlist,
    exchange="SSE",
)

print("\n最新状态：")
display(status)

# 3. 读取最近一年价格
prices = manager.store.get_daily_prices(
    ts_codes=watchlist,
    start_date="20250101",
    end_date=today_str(),
)

print("\n最近价格数据：")
display(prices.tail(10))

最近数据刷新结果：
{'instrument_count': 11, 'updated_count': 11, 'inserted_rows': 252}

最新状态：


,ts_code,latest_trade_date,expected_latest_trade_date,is_up_to_date
0,510300.SH,20260326,20260324,0
1,510500.SH,20260326,20260324,0
2,510170.SH,20260326,20260324,0
3,159915.SZ,20260326,20260324,0
4,511090.SH,20260326,20260324,0
5,511010.SH,20260326,20260324,0
6,518880.SH,20260326,20260324,0
7,159981.SZ,20260326,20260324,0
8,159985.SZ,20260326,20260324,0
9,501018.SH,20260326,20260324,0



最近价格数据：


,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount,source,created_at,updated_at
3244,518880.SH,20260313,10.852,10.880,10.791,10.802,10.938,-0.136,-1.2434,3885245.33,4.211610e+06,tushare,2026-03-24 20:53:46,2026-03-26 20:41:49
3245,518880.SH,20260316,10.630,10.700,10.605,10.664,10.802,-0.138,-1.2775,4789246.60,5.100457e+06,tushare,2026-03-24 20:53:46,2026-03-26 20:41:49
3246,518880.SH,20260317,10.632,10.696,10.630,10.637,10.664,-0.027,-0.2532,3535281.90,3.766973e+06,tushare,2026-03-24 20:53:46,2026-03-26 20:41:49
3247,518880.SH,20260318,10.638,10.639,10.570,10.606,10.637,-0.031,-0.2914,3544273.28,3.756858e+06,tushare,2026-03-24 20:53:46,2026-03-26 20:41:49
3248,518880.SH,20260319,10.283,10.345,10.080,10.126,10.606,-0.480,-4.5257,8783621.47,9.006106e+06,tushare,2026-03-24 20:53:46,2026-03-26 20:41:49
3249,518880.SH,20260320,9.900,10.066,9.883,9.896,10.126,-0.230,-2.2714,11917840.92,1.187842e+07,tushare,2026-03-24 20:53:46,2026-03-26 20:41:49
3250,518880.SH,20260323,9.488,9.500,8.906,8.941,9.896,-0.955,-9.6504,19747659.11,1.823177e+07,tushare,2026-03-24 20:53:46,2026-03-26 20:41:49
3251,518880.SH,20260324,9.210,9.303,9.105,9.299,8.941,0.358,4.0040,14905164.64,1.371674e+07,tushare,2026-03-24 20:53:46,2026-03-26 20:41:49
3252,518880.SH,20260325,9.700,9.772,9.609,9.641,9.299,0.342,3.6778,11786322.01,1.143809e+07,tushare,2026-03-25 18:44:05,2026-03-26 20:41:49
3253,518880.SH,20260326,9.628,9.637,9.358,9.450,9.641,-0.191,-1.9811,11076212.74,1.052593e+07,tushare,2026-03-26 20:41:49,2026-03-26 20:41:49


## 19. 最常用的项目初始化流程示例

这个 cell 适合项目刚开始时跑一次。

In [ ]:
from pathlib import Path
import pandas as pd
from market_data import create_manager, today_str, load_tushare_token
# Tushare Token
TUSHARE_TOKEN = load_tushare_token()


# 数据库路径：直接放在项目目录下
DB_PATH = "data/db/market_data.db"

manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date="20070101",
    default_exchange="SSE",
)

print("数据库位置：", Path(DB_PATH).resolve())
# Step 1: 初始化基础数据
manager.initialize_basic_data(
    fund_market="E",
    fund_status="L",
    cal_start_date="20070101",
    cal_end_date=today_str(),
    exchange="SSE",
)

# Step 2: 选定第一批资产池
watchlist = [
    "510300.SH",  # 沪深300ETF (核心权益：大盘蓝筹)
    "510500.SH",  # 中证500ETF (核心权益：中盘成长)
    "510170.SH",  # 50ETF (核心权益：大宗商品股票集合)
    "159915.SZ",  # 创业板ETF (核心权益：高新成长 - 纯被动)
    "511090.SH",  # 30年国债ETF (避险资产：超长债，对利率极度敏感)
    "511010.SH",  # 国债ETF (基础配置：中长期债)
    "518880.SH",  # 黄金ETF (抗通胀/避险：金属商品)
    "159981.SZ",  # 能源ETF (抗通胀：能源/电力)
    "159985.SZ",  # 豆粕ETF (抗通胀：农产品期货)
    "501018.SH",  # 南方原油LOF (抗通胀：国际原油价格)
    "515100.SH",  # 红利低波100ETF (抗通胀：红利股)
]

# Step 3: 拉取历史日线
summary = manager.update_daily_prices(
    ts_codes=watchlist,
    start_date="20070101",
    end_date=today_str(),
)

print("历史数据初始化完成：")
print(summary)

# Step 4: 检查覆盖范围
coverage = manager.check_price_coverage(ts_codes=watchlist)
display(coverage)

# Step 5: 检查是否更新到最新交易日
status = manager.check_latest_price_status(ts_codes=watchlist)
display(status)

数据库位置： D:\codeWork\RiskParity\data\db\market_data.db
历史数据初始化完成：
{'instrument_count': 10, 'updated_count': 10, 'inserted_rows': 25911}


,ts_code,start_date,end_date,has_data
0,510300.SH,20120528,20260324,1
1,510500.SH,20130315,20260324,1
2,510170.SH,20110125,20260324,1
3,159915.SZ,20111209,20260324,1
4,511090.SH,20230613,20260324,1
5,511010.SH,20130325,20260324,1
6,518880.SH,20130729,20260324,1
7,159981.SZ,20200117,20260324,1
8,159985.SZ,20191205,20260324,1
9,501018.SH,20160628,20260323,1


,ts_code,latest_trade_date,expected_latest_trade_date,is_up_to_date
0,510300.SH,20260324,20260324,1
1,510500.SH,20260324,20260324,1
2,510170.SH,20260324,20260324,1
3,159915.SZ,20260324,20260324,1
4,511090.SH,20260324,20260324,1
5,511010.SH,20260324,20260324,1
6,518880.SH,20260324,20260324,1
7,159981.SZ,20260324,20260324,1
8,159985.SZ,20260324,20260324,1
9,501018.SH,20260323,20260324,0
